# Qwen3-8B LoRA fine-tune for MAST Stage 1 (Colab)

Stage C–E of `qwen-lora-finetuning-plan.md`. Assumes `train.jsonl` / `val.jsonl`
already exist (produced locally by `stage_ab_teacher_labeling.py`) in the exact
`{"messages": [{"role": system/user/assistant, "content": ...}]}` chat format,
using the **unmodified** inference-time `STAGE1_SYS`/`STAGE1_USER` template.

**Runtime:** Runtime → Change runtime type → T4 GPU (free tier works; A100 if
you have Colab Pro — you'll likely want 2–3 runs, not one, per the plan).

## 1. Install Unsloth

Pinned install pattern Unsloth recommends for Colab, to avoid dependency drift.

In [ ]:
%%capture
import os
!pip install --upgrade pip -q
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes -q


## 2. Confirm Qwen3-8B support

Support tables change — the plan explicitly calls this out as something to verify
at run time, not trust from any written instructions. This just prints Unsloth's
version and attempts the 4-bit load below; if `FastLanguageModel.from_pretrained`
fails on the model name, check https://github.com/unslothai/unsloth#-model-support
and swap `MODEL_ID` for whatever the current 4-bit Qwen3-8B tag is.

In [ ]:
import unsloth
print("unsloth version:", unsloth.__version__)


## 3. Get your data onto the Colab VM

Easiest: mount Drive and point at wherever you uploaded `train.jsonl`/`val.jsonl`. Adjust `DATA_DIR`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DATA_DIR = Path('/content/drive/MyDrive/mast_qwen_lora')   # <-- put train.jsonl / val.jsonl here
TRAIN_PATH = DATA_DIR / 'train.jsonl'
VAL_PATH = DATA_DIR / 'val.jsonl'
OUT_DIR = DATA_DIR / 'runs' / 'qwen3_8b_lora_v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_PATH.exists(), f"missing {TRAIN_PATH} -- upload it to Drive first"
assert VAL_PATH.exists(), f"missing {VAL_PATH} -- upload it to Drive first"
print(f"train: {TRAIN_PATH}\nval:   {VAL_PATH}\nout:   {OUT_DIR}")


## 4. Load base model in 4-bit + attach LoRA

`max_seq_length=4096` per the plan (Stage D) — covers the windowed prefix + step
text comfortably. Unsloth's 4-bit dynamic quant loads directly from the HF hub;
no separate quantization step needed here (that only matters for the MLX
conversion later, in Stage E).

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_ID = "unsloth/Qwen3-8B-bnb-4bit"   # verify current tag in Unsloth's model list before running
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                      # bump to 32 if you have headroom (A100) and want more capacity
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",   # Unsloth's checkpointing variant -- extra VRAM headroom on T4
    random_state=3407,
)


## 5. Load + format the dataset

Applies Qwen3's own chat template via the tokenizer (`enable_thinking=False`,
matching every inference call in the prompted pipeline) so training sees
*exactly* the same token sequence shape as eval will. Loss is masked to the
assistant turn only (`train_on_responses_only`) — otherwise the model would
also be trained to predict the system/user prompt tokens, which wastes
capacity and dilutes the signal on the few hundred assistant tokens that
actually matter.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, train_on_responses_only

tokenizer = get_chat_template(tokenizer, chat_template="qwen3")

def formatting_func(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False, enable_thinking=False)
        for convo in examples["messages"]
    ]
    return {"text": texts}

raw_ds = load_dataset("json", data_files={"train": str(TRAIN_PATH), "val": str(VAL_PATH)})
train_ds = raw_ds["train"].map(formatting_func, batched=True)
val_ds = raw_ds["val"].map(formatting_func, batched=True)
print(train_ds)
print(val_ds)
print("\n--- sample formatted example ---\n")
print(train_ds[0]["text"][:2000])


## 6. Train

Config lifted directly from Stage D of the plan. With only ~3,000 step-examples,
**overfitting is the risk, not underfitting** — `eval_strategy="steps"` +
`eval_steps=50` so you can watch val loss diverge from train loss and stop
early if needed, rather than committing to the full 3 epochs blind.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=str(OUT_DIR / "checkpoints"),
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,     # effective batch 16
    learning_rate=1e-4,                # try 2e-4 if underfitting after this run
    num_train_epochs=3,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_seq_length=MAX_SEQ_LENGTH,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    dataset_text_field="text",
    packing=False,   # keep each step-example as its own sequence; packing would blur step boundaries
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)

# Mask the loss to assistant tokens only -- see markdown above.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)


In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)


## 6b. Sanity-check training curves

If eval_loss starts climbing while train_loss keeps falling, that's the
overfitting the plan warns about — stop there rather than letting all 3
epochs run; `load_best_model_at_end=True` above already protects you from
shipping the worst checkpoint, but look at this plot to decide whether even
fewer epochs would have been better for a re-run.

In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history
train_pts = [(e["step"], e["loss"]) for e in log if "loss" in e and "eval_loss" not in e]
eval_pts = [(e["step"], e["eval_loss"]) for e in log if "eval_loss" in e]

plt.figure(figsize=(7, 4))
if train_pts:
    xs, ys = zip(*train_pts)
    plt.plot(xs, ys, label="train_loss")
if eval_pts:
    xs, ys = zip(*eval_pts)
    plt.plot(xs, ys, label="eval_loss", marker="o")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("Qwen3-8B LoRA fine-tune")
plt.savefig(OUT_DIR / "loss_curve.png", dpi=120)
plt.show()


## 7. Quick qualitative check before merging

Generate on a handful of held-out **val** examples and eyeball that output is
still well-formed `MODE:`/`EVIDENCE:` text (plan Stage E4 — LoRA fine-tuning at
too high an LR can destabilize output formatting).

In [ ]:
FastLanguageModel.for_inference(model)

import random
random.seed(0)
sample_idxs = random.sample(range(len(val_ds)), min(5, len(val_ds)))

for i in sample_idxs:
    convo = raw_ds["val"][i]["messages"]
    prompt_msgs = convo[:2]  # system + user, drop the gold assistant turn
    inputs = tokenizer.apply_chat_template(
        prompt_msgs, tokenize=True, add_generation_prompt=True, enable_thinking=False,
        return_tensors="pt",
    ).to(model.device)
    out = model.generate(input_ids=inputs, max_new_tokens=300, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    gen_text = tokenizer.decode(out[0, inputs.shape[1]:], skip_special_tokens=True)
    print(f"=== val example {i} ===")
    print("GOLD:", convo[2]["content"][:200])
    print("PRED:", gen_text.strip()[:200])
    print()


## 8. Save adapter, merge, and export a merged HF checkpoint

This is Stage E1. You're training on CUDA/HF here but your inference pipeline
runs on MLX on Apple Silicon — the merged HF checkpoint below is the artifact
you transfer to your Mac and convert in Stage E3 (not the raw LoRA adapter).

In [ ]:
ADAPTER_DIR = OUT_DIR / "lora_adapter"
MERGED_DIR = OUT_DIR / "merged_fp16"

model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f"LoRA adapter saved to {ADAPTER_DIR}")

# merge_and_unload fuses the LoRA weights into the base model
FastLanguageModel.for_inference(model)
merged_model = model.merge_and_unload()
merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))
print(f"Merged HF checkpoint saved to {MERGED_DIR}")


## 9. Zip and download (or leave on Drive)

Since `MERGED_DIR` already lives under your mounted Drive folder, no extra
transfer step is required — just `rsync`/download that folder to your Mac.
The zip step below is only useful if you want a single-file download instead.

In [ ]:
import shutil
zip_path = shutil.make_archive(str(OUT_DIR / "merged_fp16"), "zip", root_dir=str(MERGED_DIR))
print(f"zipped: {zip_path}")

from google.colab import files
# files.download(zip_path)   # uncomment to trigger a browser download instead of using Drive


## 10. Next step (run on your Mac, not here)

Stage E3–E4 from the plan, to run locally with `mlx-lm` once `merged_fp16/`
(or its unzipped copy) is on your Mac:

```bash
pip install -U mlx-lm

mlx_lm.convert \
    --hf-path /path/to/merged_fp16 \
    --mlx-path /path/to/qwen3-8b-mast-lora-mlx \
    -q --q-bits 4

# sanity check a few steps before the full eval -- confirm MODE:/EVIDENCE: output
# is still well-formed (point mast_window_ablation.ipynb's MLX_MODEL_ID at the
# new --mlx-path and re-run a handful of trajectories)
```

Then re-point `MLX_MODEL_ID` in `mast_window_ablation.ipynb` at the new
`qwen3-8b-mast-lora-mlx` directory and re-run the window=0 / window=3 reports
on the untouched 56-trajectory eval set (plan Stage F).